# BackendOps guide

This notebook explains what `BackendOps` represents in `spacecore`, how it relates to `Context`, and how to use the predefined backends.

Current backend implementations:

- `NumpyOps` is always available.
- `JaxOps`, `CuPyOps`, and `TorchOps` are optional exports when their dependencies are installed.

This notebook uses NumPy and JAX because they are available in the development environment.


## 0. Motivation behind `BackendOps`

The library works with mathematical objects such as spaces and linear operators, but those objects still need a concrete numerical backend in order to be represented and manipulated in code.

This is precisely the role of `BackendOps`.

`BackendOps` provides a unified numerical interface through which the library performs array creation, reshaping, linear algebra, sparse operations, and related backend-dependent routines. Its purpose is not to replace the mathematical layer, but to provide the computational realization of that layer.

In other words, the library distinguishes between two complementary aspects:

* the **mathematical structure** of the objects being manipulated,
* the **numerical backend** used to store and compute with them.

This distinction becomes necessary because the same mathematical object may be represented using different array libraries. For example, one may want to work with:

* `NumPy` arrays for standard eager CPU computations,
* `JAX` arrays for JIT compilation and automatic differentiation,
* and later possibly `CuPy` or `PyTorch` arrays as well.

Without a dedicated backend abstraction, the implementation of spaces and operators would need to branch constantly on the concrete array library being used. The code would become cluttered with backend-specific logic, and the mathematical abstractions would be harder to maintain and extend.

`BackendOps` solves this problem by factoring out the backend-dependent numerical layer into one common interface. Then higher-level objects can be written against that interface rather than against a specific library.

This makes it possible for the same mathematical abstractions to work uniformly across different backends when those optional dependencies are installed.

Schematically, the design is

$$
\texttt{BackendOps} \to \texttt{Context} \to \texttt{Space} \to \texttt{LinOp}.
$$

Here:

* `BackendOps` provides the concrete numerical operations,
* `Context` selects a backend together with dtype and checking policy,
* `Space` describes the geometry of admissible objects,
* `LinOp` describes linear maps between spaces.

So the motivation behind `BackendOps` is to make the numerical implementation backend-independent while preserving a clean separation between mathematical structure and computational realization.


## 1. What `BackendOps` signifies

`BackendOps` is the backend-agnostic numerical interface used by the library internals.
In a nutshell, the principal role of `BackendOps` is to decouple operations with numerical objects from a particular `NumPy`-like library.

`BackendOps` provides a minimal set of numerical operations, and it mostly embodies a wrapper around `NumPy`-like methods.
This way, spaces never know which backend (`NumPy`, `JAX`, ...) they use.
Instead, they interact with a particular library only via `BackendOps`.

Additionally, `BackendOps` unifies methods signatures, providing minimal possible signature user should expect.
For instance, one may compare method `matmul`.
The signature of the method in `NumPy` is given by
```
    numpy.matmul(x1, x2, /, out=None, *, casting='same_kind', order='K',
                dtype=None, subok=True[, signature, axes, axis]) = <ufunc 'matmul'>,
```
while `JAX` defines
```
    jax.numpy.matmul(a, b, *, precision=None, preferred_element_type=None, out_sharding=None).
```
While the minimal part of passing two arrays is consistent, the rest of arguments with default values differ (~~though one may say they never get redefined~~).
In this case, abstract class `BackendOps` has
```
    @abstractmethod
        def matmul(self, a: DenseArray, b: DenseArray) -> DenseArray:
            ...
```
Thus, `BackendOps` demands minimum of both and giving freedom to add new arguments for the particular backend.

So `BackendOps` is not a mathematical object like a space or an operator. It is the **numerical execution interface** used by those objects.

This way, we have the following separation of roles:
- `BackendOps` says **how computations are performed**.
- `Context` says **which backend and dtype policy are active**.
- `Space` and `LinOp` use that context to store and manipulate arrays.


In [1]:
from spacecore.backend import BackendOps, Context, NumpyOps, JaxOps

numpy_ops = NumpyOps()  # Initialize BackendOps for NumPy backend
jax_ops = JaxOps()  # Initialize BackendOps for JAX backend

print(type(numpy_ops).__name__, numpy_ops.family, numpy_ops.allow_sparse)
print(type(jax_ops).__name__, jax_ops.family, jax_ops.allow_sparse)

NumpyOps numpy True
JaxOps jax True


## 2. Why the abstraction is useful

Without `BackendOps`, the rest of the library would need to branch everywhere:

- use NumPy here
- use JAX there
- use SciPy sparse in one case
- use JAX sparse in another case

That quickly becomes hard to maintain.

Instead, the library writes to the interface once. Then backend-specific classes implement the same contract.

This lets you write backend-agnostic code such as:

$$
x \mapsto \operatorname{reshape}(x), \qquad
(A, x) \mapsto Ax, \qquad
x \mapsto \operatorname{eigh}(x),
$$

without hard-coding whether the actual arrays are NumPy arrays or JAX arrays.


## 3. `BackendOps` versus `Context`

`BackendOps` only describes the backend family and the available numerical operations.
It is usually stored and retrieved via `Context` object.

`Context` adds runtime policy:

- backend ops object
- dtype
- whether checks are enabled

So in practice you will often work with `Context`, not bare `BackendOps`, when building spaces and operators.

More on `Context` see in the respective guide.

In [2]:
import numpy as np

ctx_np = Context(NumpyOps(), dtype=np.float64, enable_checks=True)
print(ctx_np)

# JAX example
# Depending on your local JAX configuration, dtype handling may differ.
ctx_jax = Context(JaxOps())
print(ctx_jax)

Context(ops=NumpyOps(family='numpy'), dtype=dtype('float64'), enable_checks=True)
Context(ops=JaxOps(family='jax'), dtype=<class 'jax.numpy.float32'>, enable_checks=True)


/Users/pavlopelikh/Documents/Dev/SpaceCore/spacecore/backend/jax/_ops.py:90: UserWarning: jax_enable_x64 is set to False, so default JAX dtype is set to float32. If you need float64, run `jax.config.update('jax_enable_x64', True)`.
  warn(


## 4. Using the predefined backends

### 4.1 NumPy backend

Use `NumpyOps` when you want standard eager CPU arrays and the simplest debugging experience.

Typical reasons to use it:

- prototyping
- debugging shapes and dtypes
- CPU-only workflows
- close interoperability with SciPy sparse matrices


In [3]:
ops = NumpyOps()

x = ops.arange(6, dtype=float)
X = ops.reshape(x, (2, 3))
I = ops.eye(3)

print(X)
print(I)
print("dense?", ops.is_dense(X))

[[0. 1. 2.]
 [3. 4. 5.]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
dense? True


### 4.2 JAX backend

Use `JaxOps` when you want the JAX execution model.

Typical reasons to use it:

- JIT compilation
- automatic differentiation
- accelerator execution
- compatibility with JAX sparse arrays

One practical point: JAX dtype behavior depends on JAX configuration, especially `jax_enable_x64`.
So `Context(JaxOps(), dtype=...)` is usually the right place to make dtype intent explicit.


In [4]:
ops = JaxOps()

x = ops.arange(6)
X = ops.reshape(x, (2, 3))
I = ops.eye(3)

print(X)
print(I)
print("dense?", ops.is_dense(X))

[[0 1 2]
 [3 4 5]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
dense? True


## 5. Write backend-agnostic code against the interface

A good pattern is to write functions that accept an `ops` object.

Then the same function can run under NumPy or JAX.


In [5]:
def gram_matrix(ops: BackendOps, A):
    """Return A^* A."""
    return ops.matmul(ops.conj(ops.transpose(A)), A)


A_np = NumpyOps().reshape(NumpyOps().arange(6, dtype=float), (3, 2))
G_np = gram_matrix(NumpyOps(), A_np)
print(G_np)

A_jax = JaxOps().reshape(JaxOps().arange(6), (3, 2))
G_jax = gram_matrix(JaxOps(), A_jax)
print(G_jax)

[[20. 26.]
 [26. 35.]]
[[20 26]
 [26 35]]


The point is that the function only depends on the abstract operations:

$$
A \mapsto A^*A.
$$

It does not need to know whether the actual backend is NumPy or JAX.


## 6. When to use `ops` directly and when to use `Context`

Use `ops` directly when:

- writing low-level backend-agnostic helper functions
- implementing internals of spaces or operators
- manipulating raw arrays in a backend-independent way

Use `Context` when:

- creating library objects
- you need dtype normalization
- you want `assert_dense`, `assert_sparse`, or `convert`
- you want one explicit runtime policy carried by objects


In [6]:
ctx = Context(NumpyOps(), dtype=np.float64, enable_checks=True)

x = ctx.asarray([1, 2, 3])
print(x, x.dtype)

# The context can also enforce dense/sparse expectations.
ctx.assert_dense(x)

[1. 2. 3.] float64


array([1., 2., 3.])

## 7. Sparse support

`BackendOps` also carries sparse-related information.

In particular:

- whether sparse arrays are supported,
- what sparse array types belong to the backend,
- how to convert to sparse form,
- how to perform sparse-dense multiplication.

This matters because the library wants the same high-level abstractions to work with both dense and sparse operator storage.


In [7]:
np_ops = NumpyOps()
jax_ops = JaxOps()

print("NumPy sparse support:", np_ops.allow_sparse)
print("NumPy sparse array dtypes:", np_ops.sparse_array)
print("\nJAX sparse support:", jax_ops.allow_sparse)
print("JAX sparse array dtypes:", jax_ops.sparse_array)

NumPy sparse support: True
NumPy sparse array dtypes: (<class 'scipy.sparse._matrix.spmatrix'>, <class 'scipy.sparse._base.sparray'>)

JAX sparse support: True
JAX sparse array dtypes: (<class 'jax.experimental.sparse.bcoo.BCOO'>, <class 'jax.experimental.sparse.bcsr.BCSR'>)


## 8. Practical advice

### Prefer this pattern

1. Choose a backend implementation (`NumpyOps` or `JaxOps`).
2. Wrap it in a `Context`.
3. Build spaces and operators from that context.

### Avoid this pattern

Do not mix raw backend arrays from different families inside the same object graph unless you explicitly convert them first.

The intended flow is:

$$
\text{raw data} \to \texttt{Context.asarray / assparse / convert} \to \text{library objects}.
$$


## 9. Future work

The current predefined backend implementations are NumPy and JAX.

Natural next extensions are:

### `CuPyOps`

This would provide a GPU-oriented array backend with a NumPy-like API, useful for users who want accelerator execution without switching to the JAX programming model.

### `TorchOps`

This would provide a PyTorch-backed implementation, useful for interoperability with existing PyTorch workflows and differentiable programming stacks.

Conceptually, both would implement the same `BackendOps` contract:

$$
\texttt{CuPyOps},\ \texttt{TorchOps} : \texttt{BackendOps}.
$$

So the abstraction is intentionally designed to make those future additions natural.


## 10. Summary

`BackendOps` is the portability layer of the library.

It signifies:

- one abstract numerical interface,
- multiple concrete backends,
- a clean separation between mathematical abstractions and execution details.

Current predefined backends:

- `NumpyOps`
- `JaxOps`

Planned future backends:

- `CuPyOps`
- `TorchOps`
